In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Dataset/matches_clean.csv", parse_dates=["match_date"])
df = df.sort_values(["match_date", "id"]).reset_index(drop=True)

In [6]:
#Reshape the df to have 1 row per team, per match

home = df[["id", "match_date", "season_start_year", "home", "home_score", "away_score"]].copy()
home.columns = ["id", "match_date", "season", "team", "gf", "ga"] 
home["venue"] = "H"

away = df[["id", "match_date", "season_start_year", "away", "away_score", "home_score"]].copy()
away.columns = ["id", "match_date", "season", "team", "gf", "ga"]
away["venue"] = "A"

#Combine the home and away dataframes into a long format dataframe, where both home and away matches are in a single df, sorted by match id and date
long = pd.concat([home, away]).sort_values(["match_date", "id"]).reset_index(drop=True)

long.shape #7967*2 checks out

(15934, 7)

In [7]:
long["pts"]= np.select([long.gf> long.ga, long.gf==long.ga], [3,1], default=0) #points earned by each row
long["win"]= (long.gf>long.ga).astype(int) #win flag

In [8]:
#Recent form (5 matches)
long["recent_5"]= long.groupby("team")["pts"].transform(
    lambda s:s.shift(1).rolling(5, min_periods=5).sum()
)

In [10]:
#average goal scored and conceded
#expanding instead of rolling, as we want to use all previous matches to calculate the average, not just the last 5
long["avg_gf"]= long.groupby("team")["gf"].transform(lambda s:s.shift(1).expanding(min_periods=5).mean())
long["avg_ga"]= long.groupby("team")["ga"].transform(lambda s:s.shift(1).expanding(min_periods=5).mean())

In [12]:
#Home win rate and away win rate
long["win_rate"]= long.groupby(["team", "venue"])["win"].transform(
    lambda s:s.shift(1).expanding(min_periods=5).mean()
)

long.tail(10)

,id,match_date,season,team,gf,ga,venue,pts,win,recent_5,avg_gf,avg_ga,win_rate
15924,605675,2022-05-15,2021,Watford,1,5,H,0,0,1.0,1.034091,1.643939,0.303030
15925,605675,2022-05-15,2021,Leicester City,5,1,A,3,1,5.0,1.416446,1.427056,0.292553
15926,605677,2022-05-15,2021,West Ham United,2,2,H,1,0,4.0,1.260997,1.473607,0.422287
15927,605677,2022-05-15,2021,Manchester City,2,2,A,1,0,15.0,1.889182,1.025066,0.453826
15928,605678,2022-05-15,2021,Wolverhampton Wanderers,1,1,H,1,0,1.0,1.079470,1.516556,0.364238
15929,605678,2022-05-15,2021,Norwich City,1,1,A,1,0,0.0,0.946970,1.829545,0.113636
15930,605669,2022-05-16,2021,Newcastle United,2,0,H,3,1,9.0,1.243056,1.456944,0.441667
15931,605669,2022-05-16,2021,Arsenal,0,2,A,0,0,12.0,1.875628,1.048995,0.462312
15932,605676,2022-05-17,2021,Southampton,1,2,H,0,0,4.0,1.237736,1.413208,0.377358
15933,605676,2022-05-17,2021,Liverpool,2,1,A,3,1,13.0,1.809045,0.963568,0.474874
